## Make dataset out of Hugginggface dataset

In [ ]:
# Cette cellule sera utilisée plus tard pour calculer le FID score
# Une fois que vous aurez généré des images avec votre modèle

In [ ]:
%pip install --upgrade transformers pillow torch torchvision datasets tqdm accelerate

In [ ]:
import os
from tqdm import tqdm
from datasets import load_dataset
import json
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

print(f"✓ Tous les modules sont chargés avec succès!")

In [ ]:
# Charger le modèle BLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Utiliser BLIP-base avec le paramètre qui évite l'erreur 404
try:
    from transformers import AutoProcessor, AutoModelForVision2Seq
    processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = AutoModelForVision2Seq.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
except:
    # Fallback sur l'ancienne méthode si disponible
    from transformers import BlipProcessor, BlipForConditionalGeneration
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

print("✓ Modèle BLIP chargé avec succès!")

def generate_caption(image):
    """Génère une caption pour une image avec BLIP"""
    # Préparer l'image
    inputs = processor(image, return_tensors="pt").to(device)
    
    # Générer la caption
    out = model.generate(**inputs, max_length=50)
    caption = processor.decode(out[0], skip_special_tokens=True)
    
    return caption

In [ ]:
dataset = load_dataset("dpdl-benchmark/oxford_flowers102")

train_dataset = dataset["train"]
fid_train_dataset = dataset["test"]
fid_test_dataset = dataset["validation"]

In [ ]:
root_dir = "./datasets/oxford_flowers102_"
images_dir = "images"
captions_dir = "captions"

images_dir_absolute = os.path.join(root_dir, images_dir)
captions_dir_absolute = os.path.join(root_dir, captions_dir)

if not os.path.exists(root_dir):
    os.makedirs(os.path.join(root_dir, images_dir))

if not os.path.exists(os.path.join(root_dir, images_dir)):
    os.makedirs(os.path.join(root_dir, images_dir))
if not os.path.exists(os.path.join(root_dir, captions_dir)):
    os.makedirs(os.path.join(root_dir, captions_dir))

image_format = "png"
json_name = "partition/data_info.json"
if not os.path.exists(os.path.join(root_dir, "partition")):
    os.makedirs(os.path.join(root_dir, "partition"))

absolute_json_name = os.path.join(root_dir, json_name)
data_info = []

order = 0
for item in tqdm(train_dataset): 
    image = item["image"]
    
    # Générer la caption avec BLIP
    caption = generate_caption(image)
    
    # Crop centré 512x512
    width, height = image.size
    left = (width - 512) // 2
    top = (height - 512) // 2
    right = left + 512
    bottom = top + 512
    
    # Si l'image est plus petite que 512x512, on la redimensionne d'abord
    if width < 512 or height < 512:
        scale = max(512 / width, 512 / height)
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.LANCZOS)
        width, height = image.size
        left = (width - 512) // 2
        top = (height - 512) // 2
        right = left + 512
        bottom = top + 512
    
    image_cropped = image.crop((left, top, right, bottom))
    
    # Sauvegarder l'image croppée
    image_cropped.save(f"{images_dir_absolute}/{order}.{image_format}")
    
    # Sauvegarder la caption générée par BLIP
    with open(f"{captions_dir_absolute}/{order}.txt", "w") as text_file:
        text_file.write(caption)
    
    width, height = 512, 512
    ratio = 1
    data_info.append({
        "height": height,
        "width": width,
        "ratio": ratio,
        "path": f"images/{order}.{image_format}",
        "prompt": caption,
    })
        
    order += 1

with open(absolute_json_name, "w") as json_file:
    json.dump(data_info, json_file)
    
print(f"Dataset created with {order} images and BLIP-generated captions!")

In [ ]:
# Sauvegarder toutes les images dans un dossier pour FID
fid_images_dir = "./datasets/oxford_flowers102_fid_images"
os.makedirs(fid_images_dir, exist_ok=True)

print("Préparation des images pour FID (crop centré 512x512)...")
for idx, item in enumerate(tqdm(fid_train_dataset)):
    image = item["image"]
    
    # Crop centré 512x512
    width, height = image.size
    
    # Calculer les coordonnées pour un crop centré
    left = (width - 512) // 2
    top = (height - 512) // 2
    right = left + 512
    bottom = top + 512
    
    # Si l'image est plus petite que 512x512, on la redimensionne d'abord
    if width < 512 or height < 512:
        # Redimensionner en gardant le ratio, puis crop
        scale = max(512 / width, 512 / height)
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.LANCZOS)
        width, height = image.size
        left = (width - 512) // 2
        top = (height - 512) // 2
        right = left + 512
        bottom = top + 512
    
    image_cropped = image.crop((left, top, right, bottom))
    image_cropped.save(f"{fid_images_dir}/{idx}.png")
    
print(f"✓ {len(fid_train_dataset)} images sauvegardées dans {fid_images_dir}")
print("\nMaintenant, calculons les statistiques FID...")

# Calculer les statistiques FID
!python -m pytorch_fid {fid_images_dir} dummy_path --save-stats

In [ ]:
# Vérifier que les images FID existent
import os
fid_dir = "./datasets/oxford_flowers102_fid_images"
num_images = len([f for f in os.listdir(fid_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"✓ {num_images} images trouvées dans {fid_dir}")

if num_images > 0:
    # Calculer les statistiques FID sur le dataset réel (test set)
    print("\nCalcul des statistiques FID sur le dataset réel...")
    os.system("python -m pytorch_fid ./datasets/oxford_flowers102_fid_images --save-stats stats_oxford_flowers.npz")
    print("\n✓ Statistiques sauvegardées dans stats_oxford_flowers.npz")
else:
    print(f"❌ Aucune image trouvée dans {fid_dir}")

In [ ]:
from tqdm import tqdm
# Sauvegarder toutes les images dans un dossier pour FID
fid_images_dir = "./datasets/oxford_flowers102_fiiid"
os.makedirs(fid_images_dir, exist_ok=True)

print("Préparation des images pour FID (crop centré 512x512)...")
for idx, item in enumerate(tqdm(fid_test_dataset)):
    image = item["image"]
    
    # Crop centré 512x512
    width, height = image.size
    
    # Calculer les coordonnées pour un crop centré
    left = (width - 512) // 2
    top = (height - 512) // 2
    right = left + 512
    bottom = top + 512
    
    # Si l'image est plus petite que 512x512, on la redimensionne d'abord
    if width < 512 or height < 512:
        # Redimensionner en gardant le ratio, puis crop
        scale = max(512 / width, 512 / height)
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.LANCZOS)
        width, height = image.size
        left = (width - 512) // 2
        top = (height - 512) // 2
        right = left + 512
        bottom = top + 512
    
    image_cropped = image.crop((left, top, right, bottom))
    image_cropped.save(f"{fid_images_dir}/{idx}.png")
    
print(f"✓ {len(fid_train_dataset)} images sauvegardées dans {fid_images_dir}")
print("\nMaintenant, calculons les statistiques FID...")


In [ ]:
!python -m pytorch_fid datasets/oxford_flowers102_fiiid dummy_path.npz
